**Подготовка датасета на основе файла final_report_2026-01-27.csv**

In [ ]:
%cd ..
%pwd

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from utils import df_date_convert, create_dataset, evaluate_model
from sales_dataset import SalesDataset
from window_features import FeatureExtractor

In [ ]:
df_csv = pd.read_csv('./etl/final_report_2026-01-27.csv')
df_csv = df_csv[(df_csv['exit_type']=='Продажа') & (df_csv['Territory'].str.contains('moscow', case=False, na=False)) ]
df_csv = df_csv[df_csv['12_2025'].notna()]

df_conv = df_date_convert(df_csv)
df_conv

In [4]:
sale_ds = SalesDataset(df_conv)
# sale_ds.all_windows_to_predict

**Обучение модели (регрессия)**

In [5]:
X, y = sale_ds.data_regress
dataset = create_dataset(X,y)

# Параметры модели
params = {
    'objective': 'regression',
    'metric':  ['rmse', 'mae'],
    'learning_rate': 0.01,
    'max_depth': 5,
    'num_leaves': 31,
    'verbosity': -1
}

# Обучение с ранней остановкой
model_early_stop = lgb.train(
    params,
    dataset.train_data,
    valid_sets=[dataset.valid_data],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(stopping_rounds=300), # Если в течение 300 итераций подряд метрика не стала лучше
        lgb.log_evaluation(50)  # Выводим логи каждые 50 итераций
    ]
)

model_early_stop.save_model("lgb_model_regress_2.txt")

Training until validation scores don't improve for 300 rounds
[50]	valid_0's rmse: 6.96556	valid_0's l1: 4.11673
[100]	valid_0's rmse: 6.13114	valid_0's l1: 3.40133
[150]	valid_0's rmse: 5.82697	valid_0's l1: 3.07263
[200]	valid_0's rmse: 5.80822	valid_0's l1: 2.92834
[250]	valid_0's rmse: 5.8945	valid_0's l1: 2.86284
[300]	valid_0's rmse: 5.95609	valid_0's l1: 2.83089
[350]	valid_0's rmse: 5.99976	valid_0's l1: 2.81904
[400]	valid_0's rmse: 6.03424	valid_0's l1: 2.81317
[450]	valid_0's rmse: 6.074	valid_0's l1: 2.8113
Early stopping, best iteration is:
[178]	valid_0's rmse: 5.78765	valid_0's l1: 2.97731


In [6]:
# Тестирование модели
# 1. Загрузка модели
loaded_model = lgb.Booster(model_file="lgb_model_regress_2.txt")

# 2. Предсказание на тестовых данных
y_pred = loaded_model.predict(dataset.X_test)
early_results = evaluate_model(dataset.y_test, y_pred, "LightGBM с ранней остановкой")

print(f"\nКоличество использованных деревьев: {model_early_stop.best_iteration}")



LightGBM с ранней остановкой Результаты:
Среднеквадратичная ошибка (MSE): 33.4969
Корень из MSE (RMSE): 5.7877
Средняя абсолютная ошибка (MAE): 2.9773
Средняя абсолютная процентная ошибка (MAPE): 69.59%
Коэффициент детерминации (R²): 0.5915

Количество использованных деревьев: 178


**Прогнозироване продаж**

In [7]:
# Получение окна продаж
res = sale_ds.get_last_window(mdlp_id=127357)
print(f'{res.month_predict=}, {res.window_sale=}\n')

# Получение фичей
ft = FeatureExtractor()
features = ft.compute_window(res.window_sale, num_month=res.month_predict)
print(features)

# Прогноз продаж
X = np.array(features.to_list(), dtype=float)
X = X.reshape(1,-1)
#
model_file = 'lgb_model_regress_2.txt'
model = lgb.Booster(model_file=model_file)
y_pred = model.predict(X)
print(f'\n{y_pred=}')

res.month_predict=1, res.window_sale=[7.0, 2.0, 3.0, 8.0, 3.0, 15.0, 5.0, 2.0, 11.0, 22.0, 7.0, 9.0]

Features(MeanW=7.833333333333333, StdW=5.683797634993311, CV=0.7255910948173041, RelLast=1.148936023540082, ZLast=0.20526178733424466, Delta1=2.0, LogRet1=0.2513143965348783, Mom3=-2.0, Mom6=-6.0, Slope10=0.7062937062937065, R2_Trend=0.1840128916655143, ZResLast=-0.5293719408218017, UpShare=0.5454545454545454, SignChanges=7.0, MaxDrawdown=0.8666666088888928, MonthCos=0.8660254037844387, MonthSin=0.49999999999999994, Lag12Target=7.0)

y_pred=array([9.08542006])
